# Simulated Price Comparison Engine (Using eBay Browse API Sandbox)

## 1. Introduction

#### This notebook will:

##### 1. Authenticate with sandbox credentials
##### 2. Perform searches via the Browse API sandbox endpoint
##### 3. Parse sample listings (sandbox-generated)
##### 4. Analyze average price, brand trends, and listing conditions
##### 5. Visualize results—all with sandbox test data only

## 2. Load Credentials (from .env)

In [ ]:
from dotenv import load_dotenv
import requests
import base64
import os

# Load environment variables from .env file
load_dotenv()

EBAY_CLIENT_ID = os.getenv("EBAY_CLIENT_ID")
EBAY_CLIENT_SECRET = os.getenv("EBAY_CLIENT_SECRET")

## 3. Authenticate (Sandbox Token)

In [ ]:
def get_sandbox_access_token():
    url = "https://api.sandbox.ebay.com/identity/v1/oauth2/token"
    auth = base64.b64encode(f"{EBAY_CLIENT_ID}:{EBAY_CLIENT_SECRET}".encode()).decode()
    headers = {
        "Content-Type": "application/x-www-form-urlencoded",
        "Authorization": f"Basic {auth}"
    }
    data = {"grant_type": "client_credentials", "scope": "https://api.ebay.com/oauth/api_scope"}
    resp = requests.post(url, headers=headers, data=data)
    #print("Status:", resp.status_code, resp.json())
    return resp.json().get("access_token")

access_token = get_sandbox_access_token()

## 4. Search via Browse API (Sandbox)

In [ ]:
def search_items(keyword, access_token, use_sandbox=True, max_items=100):
    """
    Search for items using eBay Browse API (sandbox or production).
    
    Args:
        keyword (str): Search query.
        access_token (str): OAuth token.
        use_sandbox (bool): Use sandbox API if True, production otherwise.
        max_items (int): Max number of items to fetch.

    Returns:
        List of item summaries (dicts).
    """
    base_url = "https://api.sandbox.ebay.com" if use_sandbox else "https://api.ebay.com"
    url = f"{base_url}/buy/browse/v1/item_summary/search"
    headers = {"Authorization": f"Bearer {access_token}"}
    params = {"q": keyword, "limit": 50}
    
    all_items = []
    while url and len(all_items) < max_items:
        resp = requests.get(url, headers=headers, params=params if '?' not in url else None)
        print("Status:", resp.status_code)
        if resp.status_code != 200:
            print(resp.json())
            break

        data = resp.json()
        items = data.get("itemSummaries", [])
        all_items.extend(items)

        # Follow 'next' if present and haven't reached max_items
        url = data.get("href") if use_sandbox else data.get("next", None)
        params = None  # Only needed on the first request

    return all_items[:max_items]


items = search_items("iphone", access_token, use_sandbox=True, max_items=100)
print(f"Retrieved {len(items)} items.")

## 5. Parse Listing Data

In [ ]:
import pandas as pd

def parse_sandbox_items(items):
    rows = []
    for item in items:
        # Extract category names
        categories = item.get("categories", [])
        category_names = [cat.get("categoryName") for cat in categories if "categoryName" in cat]

        rows.append({
            "itemId": item.get("itemId"),
            "title": item.get("title"),
            "price": float(item["price"]["value"]) if "price" in item else None,
            "currency": item["price"]["currency"] if "price" in item else None,
            "condition": item.get("condition"),
            "categories": category_names  # list of category names
        })
    return pd.DataFrame(rows)

df = parse_sandbox_items(items)
df.head()

## 6. Analysis & Visualizations

In [ ]:
print("Average Price:", df["price"].mean())

import seaborn as sns
import matplotlib.pyplot as plt

# Price distribution
sns.histplot(df["price"], bins=10)
plt.title("Price Distribution")

In [ ]:
# Condition distribution
sns.countplot(data=df, x="condition")
plt.title("Condition Frequency in Results");

## 7. Documentation

# Parsing eBay Sandbox Items into a DataFrame

This function `parse_sandbox_items` converts a list of item dictionaries retrieved from the eBay Sandbox Browse API into a structured pandas DataFrame.

**Features:**
- Extracts key fields such as `itemId`, `title`, `price`, `currency`, and `condition`.
- Safely parses nested `categories` field to extract a list of category names for each item.
- Handles cases where certain fields may be missing or incomplete.
- Outputs a DataFrame ideal for further analysis or visualization.

**Usage:**
Pass the raw list of items from the eBay API response to `parse_sandbox_items(items)`. The returned DataFrame contains clean, tabular data for easy inspection and manipulation.

```python
df = parse_sandbox_items(items)
df.head()